# Comparative Specifications — DiD Analysis

**Standalone notebook** — includes all data loading and feature engineering.
Place in the same folder as `merged_final_new.xlsx` and `Transposition_data.xlsx`.

## Outline
| Section | Specification | Key coefficient |
|---|---|---|
| 0 | Data loading & feature engineering | — |
| 1 | Simple OLS | `treat` |
| 2 | Difference-in-Differences | `treat:post` |
| 3 | Triple Difference (DDD) | `treat:post:impl` |
| 4 | DDD — high-dimensional FEs | `treat:post:impl` |
| 5 | Comparison table | all specs side-by-side |

## 0. Data Loading & Feature Engineering

### 0.1 Imports

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.iolib.summary2 import summary_col

BASE = Path('.').resolve()  # folder containing the data files
print(f"Working directory: {BASE}")

### 0.2 Load data

In [ ]:
df = pd.read_excel(BASE / "merged_final_new.xlsx")
df = df[df["eu27"] != "Not country group"].copy()

# Merge country implementation status
transposition = pd.read_excel(BASE / "Transposition_data.xlsx")
df = df.merge(transposition[["ipscntry","implementation_country"]], on="ipscntry", how="left")
print(f"Raw sample: {df.shape[0]:,} obs, {df.shape[1]} columns")

### 0.3 Treatment variables
`treat = 1` if firm has ≥ 250 employees **or** turnover ≥ €10 M.

In [ ]:
df["employee_treatment"] = df["scr10"].map({
    "1-9": 0, "10-49": 0, "50-249": 0, "250-499": 1, "500+": 1
})

df["turnover_treatment"] = df["scr14"].map({
    "Less than 25,000 euro": 0,
    "More than 25,000 to 50,000 euro": 0,
    "More than 50,000 to 100,000 euro": 0,
    "More than 100,000 to 250,000 euro": 0,
    "More than 250,000 to 500,000 euro": 0,
    "More than 500,000 to 2 million euro": 0,
    "More than 2 to 10 million euro": 0,
    "Don't know/No Answer": 0,
    "More than 10 to 50 million euro": 1,
    "More than 50 million euro": 1,
})

df["treat"] = np.where(
    (df["employee_treatment"] == 1) | (df["turnover_treatment"] == 1), 1,
    np.where((df["employee_treatment"] == 0) & (df["turnover_treatment"] == 0), 0, np.nan)
)
df["post"] = (df["year"] == 2024).astype(int)
df["sample_main"] = (df["implementation_country"] == 1).astype(int)

### 0.4 Outcome variables

In [ ]:
# target_engaged: 1 if firm has or plans a climate strategy
df['target_engaged'] = df['q14'].isin([
    'Yes',
    'No, but you are planning to define a strategy',
    'You are already climate neutral'
]).astype(int)

# firm_growth_ord: ordinal (-1 decreased, 0 unchanged, 1 increased)
growth_map = {'Decreased': -1, 'Remained unchanged': 0, 'Increased': 1,
              "Don't know/No Answer": np.nan}
df['firm_growth_ord'] = df['scr13a'].map(growth_map)
df['firm_growth_ord'] = df['firm_growth_ord'].fillna(df['firm_growth_ord'].mode()[0])

# firm_growth_ord_d: binary (1 = increased, 0 = flat or declined)
growth_map_d = {'Decreased': 0, 'Remained unchanged': 0, 'Increased': 1,
                "Don't know/No Answer": np.nan}
df['firm_growth_ord_d'] = df['scr13a'].map(growth_map_d)
df['firm_growth_ord_d'] = df['firm_growth_ord_d'].fillna(df['firm_growth_ord_d'].mode()[0])

### 0.5 Firm-level controls (age, sector, growth, green activities)

In [ ]:
# Firm age — ordinal and binary
age_map = {
    'Before 1 January 2014': 4, 'Before 1 January 2016': 4,
    'Between 1 January 2014 and 31 December 2016': 3,
    'Between 1 January 2016 and 31 December 2018': 3,
    'Between 1 January 2017 and 1 January 2021': 2,
    'Between 1 January 2019 and 1 January 2023': 2,
    'After 1 January 2021': 1, 'After 1 January 2023': 1,
    "Don't know/No Answer": np.nan,
    "Don't know/No Answer (DO NOT READ OUT)": np.nan
}
df['firm_age_ord'] = df['scr12'].map(age_map)
df['firm_age_ord'] = df['firm_age_ord'].fillna(df['firm_age_ord'].mode()[0])

age_map_d = {k: (1 if v >= 3 else 0) for k, v in age_map.items() if v is not np.nan}
age_map_d["Don't know/No Answer"] = np.nan
age_map_d["Don't know/No Answer (DO NOT READ OUT)"] = np.nan
df['firm_age_ord_d'] = df['scr12'].map(age_map_d)
df['firm_age_ord_d'] = df['firm_age_ord_d'].fillna(df['firm_age_ord_d'].mode()[0])

# East/West
east_west_map = {
    "Romania":0,"Poland":0,"Czechia":0,"Croatia":0,"Slovenia":0,"Hungary":0,
    "Bulgaria":0,"Latvia":0,"Estonia":0,"Lithuania":0,"Slovakia":0,
    "France":1,"Belgium":1,"Netherlands":1,"Sweden":1,"Portugal":1,"Italy":1,
    "Greece":1,"Germany":1,"Spain":1,"Austria":1,"Ireland":1,"Finland":1,
    "Denmark":1,"Luxembourg":1,"Malta":1,"Republic of Cyprus":1
}
df["east_west"] = df["ipscntry"].map(east_west_map)

# Resource efficiency action maturity (q1/q2 pairs)
def to_binary_action(series):
    if pd.api.types.is_numeric_dtype(series):
        return (series.fillna(0) != 0).astype('int8')
    s = series.astype('string').str.strip()
    return ((s.notna()) & (s != 'Not mentioned')).astype('int8')

domain_pairs = {
    'water':('q1_1','q2_1'), 'energy_saving':('q1_2','q2_2'),
    'green_energy':('q1_3','q2_3'), 'materials':('q1_4','q2_4'),
    'green_suppliers':('q1_5','q2_5'), 'waste_reduction':('q1_6','q2_6'),
    'sell_residues':('q1_7','q2_7'), 'recycling':('q1_8','q2_8'),
    'eco_design':('q1_9','q2_9'), 'other':('q1_10','q2_10')
}
for c in sorted({col for pair in domain_pairs.values() for col in pair}):
    df[c] = to_binary_action(df[c])
for name, (cur, plan) in domain_pairs.items():
    df[f'{name}_maturity_d'] = np.select(
        [(df[cur]==0)&(df[plan]==0), (df[cur]==0)&(df[plan]==1), (df[cur]==1)],
        [0, 1, 1], default=0).astype('int8')

# Green staff
s = df['dx5r'].astype(str).str.strip()
df['green_staff_any'] = np.where(s=='0 employees', 0,
    np.where(s=="Don't know/No Answer (DO NOT READ OUT)", np.nan, 1)).astype('float')

# Investment dummy
df['investment_dummy'] = (df['q4'] != 'Nothing').astype(int)

# Barriers (Q7)
q7_cols = [c for c in df.columns if c.startswith('q7_')]
for col in q7_cols:
    df[col] = np.where(df[col].isna(), 0, np.where(df[col]=='Not mentioned', 0, 1)).astype('int8')
for name, cols in [('institutional',['q7_1','q7_2','q7_3','q7_9','q7_10']),
                   ('capability',   ['q7_4','q7_6']),
                   ('market',       ['q7_5','q7_7','q7_8'])]:
    df[f'barrier_{name}']   = np.minimum(df[cols].sum(axis=1), 2)
    df[f'barrier_{name}_d'] = np.minimum(df[cols].sum(axis=1), 1)

# Support (Q8)
knowledge_cols = ['q8_1','q8_2','q8_5','q8_6']
financial_cols = ['q8_3','q8_4']
for col in knowledge_cols + financial_cols:
    df[col] = np.where(df[col].isna(), 0, np.where(df[col]=='Not mentioned', 0, 1)).astype('int8')
df['support_knowledge_d'] = np.minimum(df[knowledge_cols].sum(axis=1), 1)
df['support_financial_d'] = np.minimum(df[financial_cols].sum(axis=1), 1)
df['support_external']    = (df['q5_3'] == 'External support').astype(int)
df['support_internal']    = ((df['q5_1']=='Its own financial resources') |
                              (df['q5_2']=='Its own technical expertise')).astype(int)
df['support_internal_pr'] = ((df['q13_1']=='Its own financial resources') |
                              (df['q13_2']=='Its own technical expertise')).astype(int)

# Green market outcomes
df['green_offer_ord_d'] = df['q9'].map({
    'No and you are not planning to do so': 0,
    'No, but you are planning to do so in the next 2 years': 1,
    'Yes': 1}).fillna(0)
mapping_q10_d = {'Up to 5%':1,'6-10%':1,'11-30%':1,'31-50%':1,'51-75%':1,'More than 75%':1}
mode_q10 = df['q10'].map(mapping_q10_d).mode()[0]
mapping_q10_d["Don't know/No Answer (DO NOT READ OUT)"] = mode_q10
df['green_turnover_pct_d'] = df['q10'].map(mapping_q10_d).fillna(0)

print('Feature engineering complete.')

### 0.6 Analysis samples

In [ ]:
# Main sample: implementing countries (impl=1), complete cases
df_main  = df[df['sample_main'] == 1]
df_clean = df_main.dropna(subset=[
    "target_engaged","treat","post","scr12","nace_b","ipscntry"
]).copy()

# DDD sample: all countries (impl=0 + impl=1)
df_ddd = df.dropna(subset=[
    "target_engaged","treat","post","scr12","nace_b","ipscntry","implementation_country"
]).copy()
df_ddd["impl"] = df_ddd["implementation_country"]

# Working samples (drop residual NAs on firm_growth_ord_d)
df_s = df_clean.dropna(subset=['firm_growth_ord_d']).copy()  # OLS / DiD
df_d = df_ddd.dropna(subset=['firm_growth_ord_d']).copy()    # DDD

# Extended FE dummies for Section 4
df_d["cntry_treat"] = df_d["ipscntry"].astype(str)+"_"+df_d["treat"].astype(str)
df_d["cntry_post"]  = df_d["ipscntry"].astype(str)+"_"+df_d["post"].astype(str)

print(f"OLS/DiD sample (impl=1 only): {df_s.shape[0]:,} obs")
print(f"DDD sample (all countries):   {df_d.shape[0]:,} obs")
print(f"  impl=1 (policy countries):  {(df_d['impl']==1).sum():,}")
print(f"  impl=0 (never-treated):     {(df_d['impl']==0).sum():,}")

def cl(df_):
    """Clustered SE by country."""
    return {"cov_type":"cluster","cov_kwds":{"groups":df_["ipscntry"]}}

---
## 1. Simple OLS
Cross-sectional estimate of the firm-size (`treat`) effect with no pre/post
variation exploited. Two outcomes × two FE choices = 4 models.

| Model | Outcome | Country FE |
|---|---|---|
| 1a | `target_engaged` | No |
| 1b | `target_engaged` | Yes (`C(ipscntry)`) |
| 1c | `firm_growth_ord_d` | No |
| 1d | `firm_growth_ord_d` | Yes (`C(ipscntry)`) |

In [ ]:
s1a = smf.ols("target_engaged    ~ treat + C(nace_b) + C(firm_age_ord)",                data=df_s).fit(**cl(df_s))
s1b = smf.ols("target_engaged    ~ treat + C(nace_b) + C(firm_age_ord) + C(ipscntry)", data=df_s).fit(**cl(df_s))
s1c = smf.ols("firm_growth_ord_d ~ treat + C(nace_b) + C(firm_age_ord)",                data=df_s).fit(**cl(df_s))
s1d = smf.ols("firm_growth_ord_d ~ treat + C(nace_b) + C(firm_age_ord) + C(ipscntry)", data=df_s).fit(**cl(df_s))

for lbl, m in [("1a  target_engaged    no FE",      s1a),
               ("1b  target_engaged    country FE",  s1b),
               ("1c  firm_growth_ord_d no FE",       s1c),
               ("1d  firm_growth_ord_d country FE",  s1d)]:
    print(f"{lbl}: treat = {m.params['treat']:.4f}  (p={m.pvalues['treat']:.3f})")

---
## 2. Difference-in-Differences (DiD)
`treat:post` is the DiD coefficient — the additional change for large firms
after the policy (`post=1`, year 2024) relative to small firms.  
Sample restricted to implementing countries (`impl=1`).

| Model | Outcome | Country FE |
|---|---|---|
| 2a | `target_engaged` | No |
| 2b | `target_engaged` | Yes |
| 2c | `firm_growth_ord_d` | No |
| 2d | `firm_growth_ord_d` | Yes |

In [ ]:
s2a = smf.ols("target_engaged    ~ treat*post + C(nace_b) + C(firm_age_ord)",                data=df_s).fit(**cl(df_s))
s2b = smf.ols("target_engaged    ~ treat*post + C(nace_b) + C(firm_age_ord) + C(ipscntry)", data=df_s).fit(**cl(df_s))
s2c = smf.ols("firm_growth_ord_d ~ treat*post + C(nace_b) + C(firm_age_ord)",                data=df_s).fit(**cl(df_s))
s2d = smf.ols("firm_growth_ord_d ~ treat*post + C(nace_b) + C(firm_age_ord) + C(ipscntry)", data=df_s).fit(**cl(df_s))

for lbl, m in [("2a  target_engaged    no FE",      s2a),
               ("2b  target_engaged    country FE",  s2b),
               ("2c  firm_growth_ord_d no FE",       s2c),
               ("2d  firm_growth_ord_d country FE",  s2d)]:
    print(f"{lbl}: treat:post = {m.params['treat:post']:.4f}  (p={m.pvalues['treat:post']:.3f})")

---
## 3. Difference-in-Difference-in-Differences (DDD)
Adds `impl` (= `implementation_country`) as a third dimension.  
Countries with `impl=0` are **never-treated controls**, removing global time
trends that affect large firms everywhere regardless of policy.  
`treat:post:impl` is the DDD coefficient.

> **Note:** `C(ipscntry)` cannot be included — `impl` is constant within
> countries and is perfectly collinear with country FEs. Sector + age FEs
> are used for the 'with FE' variant.

| Model | Outcome | Controls |
|---|---|---|
| 3a | `target_engaged` | Sector only |
| 3b | `target_engaged` | Sector + firm age |
| 3c | `firm_growth_ord_d` | Sector only |
| 3d | `firm_growth_ord_d` | Sector + firm age |

In [ ]:
s3a = smf.ols("target_engaged    ~ treat*post*impl + C(nace_b)",                   data=df_d).fit(**cl(df_d))
s3b = smf.ols("target_engaged    ~ treat*post*impl + C(nace_b) + C(firm_age_ord)", data=df_d).fit(**cl(df_d))
s3c = smf.ols("firm_growth_ord_d ~ treat*post*impl + C(nace_b)",                   data=df_d).fit(**cl(df_d))
s3d = smf.ols("firm_growth_ord_d ~ treat*post*impl + C(nace_b) + C(firm_age_ord)", data=df_d).fit(**cl(df_d))

for lbl, m in [("3a  target_engaged    sector FE",      s3a),
               ("3b  target_engaged    sector+age FE",  s3b),
               ("3c  firm_growth_ord_d sector FE",      s3c),
               ("3d  firm_growth_ord_d sector+age FE",  s3d)]:
    print(f"{lbl}: treat:post:impl = {m.params['treat:post:impl']:.4f}  "
          f"(p={m.pvalues['treat:post:impl']:.3f})")

---
## 4. DDD — High-Dimensional Fixed Effects
Extends Section 3 by saturating all two-way interactions:

$$
Y_{ict} = \\beta(Post_t \\times TreatedCountry_c \\times Eligible_i)
         + \\mu_{c \\times Eligible_i}
         + \\lambda_{c \\times t}
         + \\delta_{Eligible_i \\times t}
         + X_{ic} + \\varepsilon_{ict}
$$

| Term | Implementation | Absorbs |
|---|---|---|
| $\\mu_{c \\times Eligible_i}$ | `C(cntry_treat)` | country FE + treat FE + country×treat |
| $\\lambda_{c \\times t}$ | `C(cntry_post)` | country FE + post FE + country×time |
| $\\delta_{Eligible_i \\times t}$ | `treat:post` | eligibility×time (DiD term) |

> **Warning:** Standard errors for `treat:post:impl` may be `nan` due to
> near-collinearity between `impl` (a country-group constant) and the
> country-level FEs. The coefficient direction is informative but inference
> requires caution.

| Model | Outcome |
|---|---|
| 4a | `target_engaged` |
| 4b | `firm_growth_ord_d` |

In [ ]:
s4a = smf.ols(
    """target_engaged ~ treat:post:impl
       + C(cntry_treat) + C(cntry_post) + treat:post
       + C(nace_b) + C(firm_age_ord)""",
    data=df_d
).fit(**cl(df_d))

s4b = smf.ols(
    """firm_growth_ord_d ~ treat:post:impl
       + C(cntry_treat) + C(cntry_post) + treat:post
       + C(nace_b) + C(firm_age_ord)""",
    data=df_d
).fit(**cl(df_d))

for lbl, m in [("4a  target_engaged", s4a), ("4b  firm_growth_ord_d", s4b)]:
    coef = m.params['treat:post:impl']
    se   = m.bse['treat:post:impl']
    pval = m.pvalues['treat:post:impl']
    print(f"{lbl}: treat:post:impl = {coef:.4f}  SE={se:.4f}  (p={pval:.3f})")

---
## 5. Comparison Table
All 14 models side-by-side. Only the key treatment coefficient for each
specification is shown (`drop_omitted=True`).  
Significance: \* p<0.1, \*\* p<0.05, \*\*\* p<0.01.

In [ ]:
comp_table = summary_col(
    [s1a, s1b, s1c, s1d, s2a, s2b, s2c, s2d, s3a, s3b, s3c, s3d, s4a, s4b],
    model_names=[
        "(1a)","(1b)","(1c)","(1d)",
        "(2a)","(2b)","(2c)","(2d)",
        "(3a)","(3b)","(3c)","(3d)",
        "(4a)","(4b)",
    ],
    stars=True,
    float_format="%0.4f",
    regressor_order=["treat", "treat:post", "treat:post:impl"],
    drop_omitted=True,
    info_dict={
        "N":          lambda x: f"{int(x.nobs)}",
        "R-squared":  lambda x: f"{x.rsquared:.3f}",
        "Outcome":    lambda x: x.model.endog_names,
        "Country FE": lambda x: "Yes" if any("ipscntry" in v for v in x.model.exog_names) else "No",
    },
)
print(comp_table)

In [ ]:
with open("comparison_table.txt", "w") as f:
    f.write(str(comp_table))
print("Saved: comparison_table.txt")

---
## 6. Doubly-Robust DDD

Implements the **DR DDD estimator** from Ortiz-Villavicencio & Sant'Anna (2025),
"Better Understanding Triple Differences Estimators" (OVS 2025), eq. 4.1.

### Why we need it

Sections 3–4 use the three-way fixed effects (3WFE) regression
`Y ~ treat*post*impl + FEs`.  OVS (2025) show this estimator is **generally biased** when
covariates are needed to justify the conditional DDD parallel-trends assumption (DDD-CPT).
The bias arises because 3WFE integrates covariates over the *control* distribution instead
of the *treated* distribution (paper sec. 3.1, Figure 1).

### Identification structure

In our setting a firm is **effectively treated** if it satisfies **two** criteria:

| Paper symbol | Our variable | Meaning |
|---|---|---|
| Q = 1 (eligible) | `treat = 1` | Large firm above CSRD threshold |
| S = g (enabling) | `impl = 1` | Country transposed the directive |

The never-enabling countries (`impl = 0`) serve as the comparison group (`g_c = inf`).
We use the **D1 exposure definition** (Cyprus moved to control; cutoff 2024-12-31)
from the robustness analysis in `ddd_robustness.py`.

### Estimator

The DR DDD decomposes into **three DR DiD components** (OVS 2025, eq. 4.1):

$$\widehat{ATT}_{\text{dr-ddd}} = \widehat{ATT}_{k3} + \widehat{ATT}_{k2} - \widehat{ATT}_{k1}$$

| Component | Treated group | Comparison group | Identifies |
|---|---|---|---|
| $k_3$ | impl=1, treat=1 | impl=1, treat=0 | Within-country eligibility DiD |
| $k_2$ | impl=1, treat=1 | impl=0, treat=1 | Between-country, among eligible |
| $k_1$ | impl=1, treat=1 | impl=0, treat=0 | Between-country, between eligibility |

Each component uses the **Sant'Anna & Zhao (2020) DR DiD** for repeated cross-sections:
a logistic propensity score and linear outcome regressions for each (group × period) cell.

The ATT is doubly robust: consistent if **either** the propensity score model **or**
the outcome regression model is correctly specified (not necessarily both).

Standard errors are computed from the combined **influence function**, clustered by country.

### 6.1  Covariates and exposure recoding (D1)

In [ ]:
from sklearn.linear_model import LogisticRegression, LinearRegression
from scipy import stats as scipy_stats
import warnings

# ── Covariate matrix for DR estimation ────────────────────────────────────
# One-hot sector FEs + firm age (ordinal numeric) + east/west
sector_dummies = pd.get_dummies(df_ddd['nace_b'], prefix='sec', drop_first=True)
X_cols_dr = sector_dummies.columns.tolist() + ['firm_age_ord', 'east_west']

df_dr_base = pd.concat([df_ddd.reset_index(drop=True),
                         sector_dummies.reset_index(drop=True)], axis=1)

# D1: Cyprus (transposed 2025-07-29) moved to comparison group
tr_dates = pd.read_excel('Transposition_data.xlsx')[['ipscntry', 't_date']]
tr_dates['t_date'] = pd.to_datetime(tr_dates['t_date'])
df_dr_base = df_dr_base.merge(tr_dates, on='ipscntry', how='left')

cutoff_d1 = pd.Timestamp('2024-12-31')
df_dr_base['impl_d1'] = (
    (df_dr_base['implementation_country'] == 1) &
    df_dr_base['t_date'].notna() &
    (df_dr_base['t_date'] <= cutoff_d1)
).astype(int)

# Drop rows with missing covariates
df_dr = df_dr_base.dropna(subset=X_cols_dr + ['impl_d1']).copy()
print(f'DR sample: {len(df_dr):,} obs, {df_dr["ipscntry"].nunique()} countries')
print(f'impl_d1=1 countries: {df_dr.loc[df_dr["impl_d1"]==1,"ipscntry"].nunique()}')
print(f'impl_d1=0 countries: {df_dr.loc[df_dr["impl_d1"]==0,"ipscntry"].nunique()}')

# Subgroup counts
for name, cond in [('impl=1 treat=1', (df_dr['impl_d1']==1)&(df_dr['treat']==1)),
                   ('impl=1 treat=0', (df_dr['impl_d1']==1)&(df_dr['treat']==0)),
                   ('impl=0 treat=1', (df_dr['impl_d1']==0)&(df_dr['treat']==1)),
                   ('impl=0 treat=0', (df_dr['impl_d1']==0)&(df_dr['treat']==0))]:
    print(f'  {name}: {cond.sum():,}')

### 6.2  DR DiD building block (repeated cross-sections)

In [ ]:
def dr_did_rc(Y, post, treated, X_mat, cluster):
    """
    Doubly-robust DiD for repeated cross-sections.
    Sant'Anna & Zhao (2020), adapted as a building block for the DR DDD.

    Parameters
    ----------
    Y       : outcome (1-D array, length n_k)
    post    : 1=post period, 0=pre (1-D array, length n_k)
    treated : 1=treated group, 0=comparison group (1-D array, length n_k)
    X_mat   : covariate matrix WITHOUT intercept (n_k x p)
    cluster : cluster labels for SE (1-D array, length n_k)

    Returns
    -------
    att : float  -- DR DiD point estimate
    psi : array  -- influence function (length n_k), mean = att
    """
    n_k = len(Y)

    # 1. Propensity score: P(treated=1 | X) via logistic regression
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        lr = LogisticRegression(max_iter=2000, solver='lbfgs', C=1e6)
        lr.fit(X_mat, treated)
    ps = lr.predict_proba(X_mat)[:, 1]
    ps = np.clip(ps, 1e-6, 1 - 1e-6)

    # 2. Outcome regressions: E[Y | comparison_group, period, X]
    for prd in [1, 0]:
        mask = (treated == 0) & (post == prd)
        if mask.sum() < 5:
            raise ValueError(f'Too few comparison obs in period {prd}: {mask.sum()}')
        coef = LinearRegression().fit(X_mat[mask], Y[mask])
        if prd == 1:
            m_post = coef.predict(X_mat)
        else:
            m_pre  = coef.predict(X_mat)

    # Cell-specific regression prediction
    m_ctrl = post * m_post + (1 - post) * m_pre
    r = Y - m_ctrl  # regression residuals

    # 3. Riesz representers (unnormalized)
    w_tp = treated * post                                      # treated, post
    w_tr = treated * (1 - post)                               # treated, pre
    w_cp = (1 - treated) * post        * ps / (1 - ps)       # ctrl,    post (IPW)
    w_cr = (1 - treated) * (1 - post)  * ps / (1 - ps)       # ctrl,    pre  (IPW)

    E_tp = w_tp.mean(); E_tr = w_tr.mean()
    E_cp = w_cp.mean(); E_cr = w_cr.mean()

    # 4. DR ATT = (att_trt_post - att_trt_pre) - (att_ctrl_post - att_ctrl_pre)
    att_tp = (w_tp * r).mean() / E_tp
    att_tr = (w_tr * r).mean() / E_tr
    att_cp = (w_cp * r).mean() / E_cp
    att_cr = (w_cr * r).mean() / E_cr
    att = (att_tp - att_tr) - (att_cp - att_cr)

    # 5. Influence function (uncentered; mean over n_k = att)
    psi = (w_tp / E_tp - w_tr / E_tr) * r - (w_tp / E_tp) * att_tp + (w_tr / E_tr) * att_tr \
        - (w_cp / E_cp - w_cr / E_cr) * r + (w_cp / E_cp) * att_cp - (w_cr / E_cr) * att_cr

    return att, psi

### 6.3  DR DDD assembler

In [ ]:
def dr_ddd(df, outcome, impl_col, treat_col, post_col, X_cols, cluster_col):
    """
    Doubly-robust DDD estimator.

    DDD = ATT_k3 + ATT_k2 - ATT_k1, each a DR DiD (OVS 2025 eq. 4.1).

    Comparison groups vs treated (impl=1, treat=1):
      k3: (impl=1, treat=0)  within-country, between eligibility
      k2: (impl=0, treat=1)  between countries, among eligible
      k1: (impl=0, treat=0)  between countries, between eligibility

    SE from combined influence function, clustered by country.
    """
    T_mask = (df[impl_col] == 1) & (df[treat_col] == 1)
    comp = {
        'k3': (df[impl_col] == 1) & (df[treat_col] == 0),
        'k2': (df[impl_col] == 0) & (df[treat_col] == 1),
        'k1': (df[impl_col] == 0) & (df[treat_col] == 0),
    }

    n = len(df)
    post    = df[post_col].to_numpy().astype(float)
    Y       = df[outcome].to_numpy().astype(float)
    cluster = df[cluster_col].to_numpy()
    X_mat   = df[X_cols].to_numpy().astype(float)

    atts, psis_full = {}, {}
    for name, C_mask in comp.items():
        sel = (T_mask | C_mask).to_numpy()
        n_k = sel.sum()
        treated_flag = T_mask.to_numpy()[sel].astype(int)
        att_k, psi_k = dr_did_rc(
            Y[sel], post[sel], treated_flag, X_mat[sel], cluster[sel]
        )
        atts[name] = att_k
        # Extend psi to full n (zero for non-members), scale by n/n_k so
        # mean over full n equals att_k (OVS 2025 influence-function aggregation).
        psi_full = np.zeros(n)
        psi_full[sel] = psi_k * (n / n_k)
        psis_full[name] = psi_full

    ddd_att = atts['k3'] + atts['k2'] - atts['k1']

    # Combined DDD influence function
    psi_ddd = psis_full['k3'] + psis_full['k2'] - psis_full['k1']

    # Clustered SE (Cameron-Gelbach-Miller sandwich)
    cluster_vals = np.unique(cluster)
    G = len(cluster_vals)
    psi_c = psi_ddd - ddd_att  # centre around att for the sandwich
    cluster_sums = np.array([psi_c[cluster == c].sum() for c in cluster_vals])
    se = np.sqrt(G / (G - 1) * cluster_sums @ cluster_sums) / n

    t_stat = ddd_att / se
    p_val  = 2 * scipy_stats.norm.sf(abs(t_stat))

    return {
        'att': ddd_att, 'se': se, 't': t_stat, 'p': p_val,
        'ci_low':  ddd_att - 1.96 * se,
        'ci_high': ddd_att + 1.96 * se,
        'n': n, 'G': G,
        'components': {k: round(v, 6) for k, v in atts.items()},
        'psi': psi_ddd, 'cluster': cluster,
    }

### 6.4  Estimates for both outcomes

In [ ]:
OUTCOMES_DR = {
    'target_engaged':    'Climate target engagement',
    'firm_growth_ord_d': 'Firm growth (binary)',
}

dr_results = {}
for col, label in OUTCOMES_DR.items():
    df_run = df_dr.dropna(subset=[col]).copy()
    result = dr_ddd(
        df_run,
        outcome     = col,
        impl_col    = 'impl_d1',
        treat_col   = 'treat',
        post_col    = 'post',
        X_cols      = X_cols_dr,
        cluster_col = 'ipscntry',
    )
    dr_results[col] = result
    print(f'\n{label}')
    print(f'  DR DDD ATT : {result["att"]:+.4f}')
    print(f'  SE (cluster): {result["se"]:.4f}')
    print(f'  t-stat      : {result["t"]:+.3f}')
    print(f'  p-value     : {result["p"]:.3f}')
    print(f'  95% CI      : [{result["ci_low"]:+.4f}, {result["ci_high"]:+.4f}]')
    print(f'  N obs       : {result["n"]:,}  |  G clusters: {result["G"]}')
    print(f'  Components  : k3={result["components"]["k3"]:+.4f}  ',
          f'k2={result["components"]["k2"]:+.4f}  ',
          f'k1={result["components"]["k1"]:+.4f}')

### 6.5  DR DDD vs 3WFE DDD — comparison

The 3WFE coefficients below come from the baseline spec (D0, `treat*post*impl + sector + age FEs`,
analytic cluster-robust SE).  They are reproduced from Section 3 for comparison.

In [ ]:
def stars(p):
    return '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.10 else ''

# 3WFE DDD references from Section 3 (sector+age FE, analytic SE)
ref_3wfe = {
    'target_engaged':    (s3b.params['treat:post:impl'], s3b.pvalues['treat:post:impl']),
    'firm_growth_ord_d': (s3d.params['treat:post:impl'], s3d.pvalues['treat:post:impl']),
}

print('=' * 78)
print(f'{"Outcome":<22} {"Estimator":<20} {"ATT":>9} {"SE":>8} {"p":>8} {"CI 95%":>22}')
print('-' * 78)

for col, label in OUTCOMES_DR.items():
    dr  = dr_results[col]
    c3, p3 = ref_3wfe[col]
    se3 = s3b.bse['treat:post:impl'] if col == 'target_engaged' else s3d.bse['treat:post:impl']
    ci3_lo = c3 - 1.96 * se3;  ci3_hi = c3 + 1.96 * se3

    print(f'{label:<22} {"3WFE DDD (sec.3)":<20} {c3:>+9.4f} {se3:>8.4f}',
          f'{p3:>6.3f}{stars(p3):<2} [{ci3_lo:+.4f}, {ci3_hi:+.4f}]')
    print(f'{"":<22} {"DR DDD (sec.6)":<20} {dr["att"]:>+9.4f} {dr["se"]:>8.4f}',
          f'{dr["p"]:>6.3f}{stars(dr["p"]):<2} [{dr["ci_low"]:+.4f}, {dr["ci_high"]:+.4f}]')
    print()

print('=' * 78)
print('SE: 3WFE uses analytic cluster-robust SE; DR DDD uses influence-function cluster SE.')
print('3WFE relies on UNCONDITIONAL DDD-CPT; DR DDD is valid under CONDITIONAL DDD-CPT.')
print('Exposure: 3WFE uses D0 (original binary); DR DDD uses D1 (Cyprus moved to control).')

### 6.6  Save DR DDD results

In [ ]:
rows = []
for col, label in OUTCOMES_DR.items():
    r = dr_results[col]
    rows.append({
        'outcome': col, 'label': label,
        'att': r['att'], 'se': r['se'], 't': r['t'], 'p': r['p'],
        'ci_low': r['ci_low'], 'ci_high': r['ci_high'],
        'n': r['n'], 'G': r['G'],
        'att_k3': r['components']['k3'],
        'att_k2': r['components']['k2'],
        'att_k1': r['components']['k1'],
    })

dr_table = pd.DataFrame(rows)
dr_table.to_csv('dr_ddd_results.csv', index=False)
print('Saved: dr_ddd_results.csv')
print(dr_table.to_string(index=False))